# Preprocesado de Audios - Guitarra, Acordeón y Batería

## Sofia Gonzalez - Mateo Llanos - Nicolas Hurtado

Este notebook extrae características MFCC de todos los audios del dataset y construye dos matrices:
- **Matriz 1** (`X_frames`): Todos los frames de todos los audios concatenados → forma (~N_frames_total, 36)
- **Matriz 2** (`X_resumen`): Un vector de 72 valores por audio (promedio + desviación estándar) → forma (3900, 72)

## Imports y configuración

In [1]:
import numpy as np
import librosa
from scipy.fftpack import dct
from pathlib import Path
import time

RUTA_DATASET = Path("Dataset") / "Audios"

INSTRUMENTOS = {
    "Guitarra":  0,
    "Acordeon":  1,
    "Bateria":   2
}

# Parámetros de extracción
TAMANO_CUADRO  = 0.025   # 25 ms
PASO_CUADRO    = 0.010   # 10 ms
NFFT           = 512
NUM_FILTROS    = 40
NUM_CEPS       = 12      # 12 MFCC → 12 delta → 12 delta2 = 36 total
PREENFASIS     = 0.97

print("Configuración lista.")

Configuración lista.


## Función de extracción de características para UN audio

In [2]:
def extraer_caracteristicas(ruta_audio):
    """
    Recibe la ruta de un archivo .wav y devuelve:
      - carac_norm: matriz (num_frames, 36) normalizada con CMVN
    Retorna None si hay algún error al leer el archivo.
    """
    try:
        # --- Carga ---
        y, sr = librosa.load(ruta_audio, sr=None)

        # --- Preénfasis ---
        y_pre = np.append(y[0], y[1:] - PREENFASIS * y[:-1])

        # --- Ventaneo / Framing ---
        long_cuadro   = int(TAMANO_CUADRO * sr)
        paso_muestras = int(PASO_CUADRO   * sr)
        long_senal    = len(y_pre)

        num_cuadros = int(np.ceil(float(np.abs(long_senal - long_cuadro)) / paso_muestras))

        # Rellenar con ceros para que todos los frames tengan el mismo tamaño
        long_rellena  = num_cuadros * paso_muestras + long_cuadro
        senal_rellena = np.append(y_pre, np.zeros(long_rellena - long_senal))

        # Construir matriz de frames
        indices = (np.tile(np.arange(0, long_cuadro), (num_cuadros, 1))
                   + np.tile(np.arange(0, num_cuadros * paso_muestras, paso_muestras), (long_cuadro, 1)).T)
        cuadros = senal_rellena[indices]

        # Ventana de Hamming
        cuadros *= np.hamming(long_cuadro)

        # --- FFT y espectro de potencia ---
        magnitud   = np.abs(np.fft.fft(cuadros, NFFT))
        potencia   = (1.0 / NFFT) * (magnitud ** 2)

        # --- Banco de filtros Mel ---
        mel_baja = 0
        mel_alta = 2595 * np.log10(1 + (sr / 2) / 700)
        puntos_mel = np.linspace(mel_baja, mel_alta, NUM_FILTROS + 2)
        puntos_hz  = 700 * (10 ** (puntos_mel / 2595) - 1)
        bins = np.floor((NFFT + 1) * puntos_hz / sr).astype(int)

        banco_filtros = np.zeros((NUM_FILTROS, int(np.floor(NFFT))))
        for m in range(1, NUM_FILTROS + 1):
            fi, fc, fd = bins[m-1], bins[m], bins[m+1]
            for k in range(fi, fc):
                banco_filtros[m-1, k] = (k - bins[m-1]) / (bins[m] - bins[m-1])
            for k in range(fc, fd):
                banco_filtros[m-1, k] = (bins[m+1] - k) / (bins[m+1] - bins[m])

        energia = np.dot(potencia, banco_filtros.T)
        energia = np.where(energia == 0, np.finfo(float).eps, energia)
        energia = 20 * np.log10(energia)

        # --- DCT → MFCC ---
        mfcc = dct(energia, type=2, axis=1, norm='ortho')[:, :NUM_CEPS]

        # --- Deltas ---
        delta_mfcc  = librosa.feature.delta(mfcc, order=1, width=9)
        delta2_mfcc = librosa.feature.delta(mfcc, order=2, width=9)

        # --- Concatenar → (num_frames, 36) ---
        caracteristicas = np.concatenate((mfcc, delta_mfcc, delta2_mfcc), axis=1)

        # --- Normalización CMVN ---
        std = np.std(caracteristicas, axis=0)
        std[std == 0] = np.finfo(float).eps   # evitar división por cero
        carac_norm = (caracteristicas - np.mean(caracteristicas, axis=0)) / std

        return carac_norm

    except Exception as e:
        print(f"  ⚠ Error en {ruta_audio.name}: {e}")
        return None


print("Función lista.")

Función lista.


## Procesamiento de todos los audios

Aquí se procesan los 3 instrumentos y se llenan las dos estructuras:
- `lista_frames`: acumula los frames de cada audio (para Matriz 1)
- `lista_resumen`: acumula el vector de 72 por audio (para Matriz 2)

In [3]:
# Listas temporales para ir acumulando
lista_frames  = []   # cada elemento es (num_frames, 36)
labels_frames = []   # etiqueta repetida por cada frame

lista_resumen  = []  # cada elemento es un vector de 72
labels_resumen = []  # etiqueta por audio

tiempo_inicio = time.time()

for nombre_instrumento, etiqueta in INSTRUMENTOS.items():
    carpeta = RUTA_DATASET / nombre_instrumento
    archivos = sorted(carpeta.glob("*.wav"))

    print(f"\n{'='*50}")
    print(f"Procesando: {nombre_instrumento} | Archivos encontrados: {len(archivos)}")
    print(f"{'='*50}")

    for i, archivo in enumerate(archivos):
        carac = extraer_caracteristicas(archivo)

        if carac is None:
            continue  # saltar audios con error

        # ── Matriz 1: guardar todos los frames ──────────────────
        lista_frames.append(carac)                          # (num_frames, 36)
        labels_frames.extend([etiqueta] * carac.shape[0])  # repetir etiqueta

        # ── Matriz 2: resumir en 72 valores ─────────────────────
        promedio  = np.mean(carac, axis=0)   # vector de 36
        std_dev   = np.std(carac,  axis=0)   # vector de 36
        resumen   = np.concatenate([promedio, std_dev])  # vector de 72

        lista_resumen.append(resumen)
        labels_resumen.append(etiqueta)

        # Progreso cada 100 audios
        if (i + 1) % 100 == 0:
            print(f"  [{i+1}/{len(archivos)}] {archivo.name} procesado")

    print(f"  ✓ {nombre_instrumento} completado.")

print(f"\nTiempo total: {(time.time()-tiempo_inicio)/60:.1f} minutos")


Procesando: Guitarra | Archivos encontrados: 1300


d:\nicolas\University\10 Semestre\Aprendizaje Automatico\MachineLearning_MusicInstruments\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  [100/1300] 1088.wav procesado
  [200/1300] 1178.wav procesado
  [300/1300] 1268.wav procesado
  [400/1300] 175.wav procesado
  [500/1300] 274.wav procesado
  [600/1300] 368.wav procesado
  [700/1300] 458.wav procesado
  [800/1300] 548.wav procesado
  [900/1300] 638.wav procesado
  [1000/1300] 728.wav procesado
  [1100/1300] 818.wav procesado
  [1200/1300] 908.wav procesado
  [1300/1300] 999.wav procesado
  ✓ Guitarra completado.

Procesando: Acordeon | Archivos encontrados: 1300
  [100/1300] 2619.wav procesado
  [200/1300] 2719.wav procesado
  [300/1300] 2819.wav procesado
  [400/1300] 2919.wav procesado
  [500/1300] 3019.wav procesado
  [600/1300] 3119.wav procesado
  [700/1300] 3220.wav procesado
  [800/1300] 3321.wav procesado
  [900/1300] 3421.wav procesado
  [1000/1300] 3521.wav procesado
  [1100/1300] 3621.wav procesado
  [1200/1300] 908.wav procesado
  [1300/1300] 999.wav procesado
  ✓ Acordeon completado.

Procesando: Bateria | Archivos encontrados: 1300
  [100/1300] 2448.wav

## Construir las matrices finales

In [4]:
# ── Matriz 1: apilar todos los frames verticalmente ──────────────
X_frames  = np.vstack(lista_frames)           # (~N_frames_total, 36)
y_frames  = np.array(labels_frames)           # (~N_frames_total,)

# ── Matriz 2: cada fila es un audio, 72 columnas ─────────────────
X_resumen = np.array(lista_resumen)           # (num_audios, 72)
y_resumen = np.array(labels_resumen)          # (num_audios,)

print("MATRIZ 1 - Frames completos:")
print(f"  X_frames  → {X_frames.shape}   (filas=frames totales, cols=36 características)")
print(f"  y_frames  → {y_frames.shape}")

print("\nMATRIZ 2 - Resumen por audio:")
print(f"  X_resumen → {X_resumen.shape}   (filas=audios, cols=72: 36 medias + 36 std)")
print(f"  y_resumen → {y_resumen.shape}")

print("\nDistribución de clases en X_resumen:")
for nombre, etiqueta in INSTRUMENTOS.items():
    n = np.sum(y_resumen == etiqueta)
    print(f"  {nombre} (clase {etiqueta}): {n} audios")

MATRIZ 1 - Frames completos:
  X_frames  → (1166100, 36)   (filas=frames totales, cols=36 características)
  y_frames  → (1166100,)

MATRIZ 2 - Resumen por audio:
  X_resumen → (3900, 72)   (filas=audios, cols=72: 36 medias + 36 std)
  y_resumen → (3900,)

Distribución de clases en X_resumen:
  Guitarra (clase 0): 1300 audios
  Acordeon (clase 1): 1300 audios
  Bateria (clase 2): 1300 audios


## Guardar las matrices en disco

Se guardan en formato `.npz` (NumPy comprimido).

In [5]:
RUTA_SALIDA = Path("Dataset") / "Caracteristicas"
RUTA_SALIDA.mkdir(parents=True, exist_ok=True)

# Guardar Matriz 1
np.savez_compressed(
    RUTA_SALIDA / "matriz_frames.npz",
    X=X_frames,
    y=y_frames
)

# Guardar Matriz 2
np.savez_compressed(
    RUTA_SALIDA / "matriz_resumen.npz",
    X=X_resumen,
    y=y_resumen
)

print(f"✓ Archivos guardados en: {RUTA_SALIDA.resolve()}")
print(f"  - matriz_frames.npz")
print(f"  - matriz_resumen.npz")

✓ Archivos guardados en: D:\nicolas\University\10 Semestre\Aprendizaje Automatico\MachineLearning_MusicInstruments\Dataset\Caracteristicas
  - matriz_frames.npz
  - matriz_resumen.npz


In [6]:
import pandas as pd

INSTRUMENTOS_NOMBRES = {0: "Guitarra", 1: "Acordeon", 2: "Bateria"}

# ── CSV de Matriz 2 (resumen por audio - 72 columnas) ─────────────
# Crear nombres de columnas descriptivos
columnas_media = [f"media_coef{i+1}" for i in range(36)]
columnas_std   = [f"std_coef{i+1}"   for i in range(36)]
columnas       = columnas_media + columnas_std

df_resumen = pd.DataFrame(X_resumen, columns=columnas)
df_resumen["etiqueta"]    = y_resumen
df_resumen["instrumento"] = df_resumen["etiqueta"].map(INSTRUMENTOS_NOMBRES)

df_resumen.to_csv(RUTA_SALIDA / "matriz_resumen.csv", index=False)
print(f"✓ matriz_resumen.csv guardado → {df_resumen.shape[0]} filas x {df_resumen.shape[1]} columnas")


# ── CSV de Matriz 1 (frames - advertencia: archivo muy grande) ────
columnas_frames = [f"coef{i+1}" for i in range(36)]

df_frames = pd.DataFrame(X_frames, columns=columnas_frames)
df_frames["etiqueta"]    = y_frames
df_frames["instrumento"] = df_frames["etiqueta"].map(INSTRUMENTOS_NOMBRES)

df_frames.to_csv(RUTA_SALIDA / "matriz_frames.csv", index=False)
print(f"✓ matriz_frames.csv guardado  → {df_frames.shape[0]} filas x {df_frames.shape[1]} columnas")

print("\nVista previa de matriz_resumen (primeras 5 filas):")
df_resumen.head()

✓ matriz_resumen.csv guardado → 3900 filas x 74 columnas


KeyboardInterrupt: 

## Si queremos usar las matrices en el futuro

Solo es correr este bloque en cualquier otro notebook para recuperar las matrices sin reprocesar:

In [ ]:
# ── CARGA RÁPIDA (usar en futuros notebooks) ──────────────────────
from pathlib import Path
import numpy as np

RUTA_SALIDA = Path("Dataset") / "Caracteristicas"

# Cargar Matriz 1
datos_frames  = np.load(RUTA_SALIDA / "matriz_frames.npz")
X_frames  = datos_frames["X"]
y_frames  = datos_frames["y"]

# Cargar Matriz 2
datos_resumen = np.load(RUTA_SALIDA / "matriz_resumen.npz")
X_resumen = datos_resumen["X"]
y_resumen = datos_resumen["y"]

print("Matrices cargadas correctamente:")
print(f"  X_frames:  {X_frames.shape}")
print(f"  X_resumen: {X_resumen.shape}")

Matrices cargadas correctamente:
  X_frames:  (1166100, 36)
  X_resumen: (3900, 72)
